In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import pandas as pd
from PIL import Image
import torchvision.models as models
from tqdm import tqdm
from torch.nn.functional import softmax, one_hot, log_softmax
import numpy as np
import matplotlib.cm as cm




In [2]:
train = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/train.csv")
test = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/test.csv")
classes = train["expert_consensus"].unique()
mapping = {c: i for i, c in enumerate(classes)}
num_classes = classes.shape[0]

vote_cols = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]
train_votes = train[vote_cols].astype(float)

class_totals = train_votes.sum()
vote_dist = (class_totals / class_totals.sum()).values  # (6,)

consensus_counts = train["expert_consensus"].value_counts()
consensus_dist = np.array(
    [consensus_counts.get(col.split("_")[0], 0) for col in vote_cols], dtype=float
)
if consensus_dist.sum() > 0:
    consensus_dist = consensus_dist / consensus_dist.sum()
else:
    consensus_dist = np.full_like(consensus_dist, 1.0 / len(consensus_dist))

# Use a stronger weight on the empirical vote distribution (0.8) and a smaller weight on the consensus prior (0.2)
fallback_probs = 0.8 * vote_dist + 0.2 * consensus_dist  # (6,)

patient_vote_sums = train.groupby("patient_id")[vote_cols].sum()
patient_totals = patient_vote_sums.sum(axis=1).replace(0, np.nan)  # avoid div by zero
patient_probs_df = patient_vote_sums.div(patient_totals, axis=0)

fallback_series = pd.Series(fallback_probs, index=vote_cols)
patient_probs_df = patient_probs_df.fillna(fallback_series)

patient_probs = {
    pid: row.values.astype(float) for pid, row in patient_probs_df.iterrows()
}




In [3]:
cmap = cm.get_cmap("viridis")




/tmp/ipykernel_55/3521101325.py:1: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap("viridis")


In [4]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        specto_id = self.data.loc[idx, "spectrogram_id"]
        specto_path = f"/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/{specto_id}.parquet"
        specto = pd.read_parquet(specto_path)
        spectrogram = Image.fromarray((cmap(specto) * 255).astype(np.uint8))
        if self.transform:
            spectrogram = self.transform(spectrogram)[:3, :, :]
        return spectrogram




In [5]:
transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
    ]
)




In [6]:
test_dataset = ImageDataset(test, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)




In [7]:
class ResNet(nn.Module):
    def __init__(self, num_classes):
        super(ResNet, self).__init__()
        self.resnet = models.resnet18(pretrained=False)
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.resnet(x)
        x = softmax(x, dim=1)
        return x




In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet(num_classes).to(device)

model_path = "/kaggle/input/hms-train-spectogram-images/trained_model.pt"
model_loaded = False
try:
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model_loaded = True
except Exception as e:
    print(f"Model loading failed ({e}); will use fallback probabilities.")




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model loading failed ([Errno 2] No such file or directory: '/kaggle/input/hms-train-spectogram-images/trained_model.pt'); will use fallback probabilities.


In [9]:
if model_loaded:
    model.eval()
    out = []
    pbar = tqdm(test_loader)
    for images in pbar:
        images = images.to(device)
        with torch.no_grad():
            outputs_batch = model(images)
        out.append(outputs_batch.cpu().numpy())
    outputs = np.vstack(out)  # (num_test, num_classes)
else:
    fallback_array = np.empty((len(test), 6), dtype=float)
    for i, pid in enumerate(test["patient_id"]):
        fallback_array[i] = patient_probs.get(pid, fallback_probs)
    outputs = fallback_array




In [10]:
# Normalise rows to sum to 1
row_sums = outputs.sum(axis=1, keepdims=True)
outputs = outputs / np.where(row_sums == 0, 1, row_sums)

# Add tiny epsilon smoothing to avoid exact zeros (helps KL divergence)
epsilon = 1e-6
outputs = outputs + epsilon
outputs = outputs / outputs.sum(axis=1, keepdims=True)




In [11]:
submission = test[["eeg_id"]].copy()
submission["seizure_vote"] = outputs[:, 0]
submission["lpd_vote"] = outputs[:, 1]
submission["gpd_vote"] = outputs[:, 2]
submission["lrda_vote"] = outputs[:, 3]
submission["grda_vote"] = outputs[:, 4]
submission["other_vote"] = outputs[:, 5]




In [12]:
submission.to_csv("submission.csv", index=False)
submission

,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,2578018731,0.001617,1.616252e-02,0.564846,6.061570e-03,1.850504e-01,2.262623e-01
1,2578018731,0.001617,1.616252e-02,0.564846,6.061570e-03,1.850504e-01,2.262623e-01
2,2578018731,0.001617,1.616252e-02,0.564846,6.061570e-03,1.850504e-01,2.262623e-01
3,2578018731,0.001617,1.616252e-02,0.564846,6.061570e-03,1.850504e-01,2.262623e-01
4,1706196508,0.114687,3.003292e-01,0.005777,1.386140e-01,1.980286e-02,4.207906e-01
...,...,...,...,...,...,...,...
9845,3349371726,0.959995,9.999940e-07,0.040001,9.999940e-07,9.999940e-07,9.999940e-07
9846,641615525,0.959995,9.999940e-07,0.040001,9.999940e-07,9.999940e-07,9.999940e-07
9847,641615525,0.959995,9.999940e-07,0.040001,9.999940e-07,9.999940e-07,9.999940e-07
9848,641615525,0.959995,9.999940e-07,0.040001,9.999940e-07,9.999940e-07,9.999940e-07
